# 05_08 Batch B: an uncontaminated trailing level for post-event weeks

Third feature iteration on the two-stage LightGBM model, continuing from `05_07`.
**One mechanism, two columns**, refitting only the two-stage model. The comparison
baseline is the `05_07` shrinkage run, backed up in
`reports/results/backup_position_lift_shrinkage/`.

Target metrics as before: pooled WAPE at row / article-store-week / article-day
grains plus relative bias; working goal WAPE ≤ 30% at the article-store-week grain
with bias near zero.

## Step-by-step: what was done and why

**Step 1 — The problem.** After the event underforecasts were reduced (05_06,
05_07), the worst remaining Pseudo 890 origin is the **week after Easter**
(2026-04-13: WAPE 33.9%, forecast/actual 1.289 — a level *over*forecast).
`05_04` §8 already located the mechanism: the `rolling_24_mean` feature entering
the 13 April origin is inflated relative to a pre-run-up baseline for most
series, because the trailing 24 *active* rows still contain the extreme
pre-Easter stock-up days. The same echo touches every week within roughly a
month after a major event window (post-May-1, post-Pentecost), and its mirror
image — a quiet post-holiday lull inside the window — can also bias the level
estimate.

**Step 2 — Why the model cannot fix this alone.** The boosters receive
`rolling_24_mean` and `rolling_6_mean` but no way to know *how much* of those
windows was event-distorted: the trailing windows silently mix ordinary and
event-window days in proportions the model cannot observe.

**Step 3 — The fix: two columns, no changes to existing features.**

- `rolling_24_mean_non_event` — the mean over the final 24 active rows whose
  calendar `holiday_event_window` is `none`: the same level estimate the model
  already trusts, but with run-up and echo days excluded before the window is
  formed.
- `event_window_share_last_24` — the share of the standard trailing 24 active
  rows that fall inside an event window: an explicit contamination signal, so a
  tree can split on "trailing mean is mostly event days" and switch to the clean
  estimate.

Both features go to both stages (no routing exclusions); leakage rules are
unchanged (strictly pre-origin, active rows only).

**Step 4 — Rebuild and refit.** `FEATURE_BUILDER_VERSION` bumped
(2026-08-06.20 → .21), all 72 partitions rebuilt, only the two-stage model refit
with unchanged hyperparameters. A unit test verifies the non-event mean ignores
inflated event-window demand while the standard mean does not.

In [1]:
from pathlib import Path
import json
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design
from src.models.lightgbm import TWO_STAGE_MODEL_NAME
from src.models.lightgbm.features.builder import _holiday_calendar
from src.models.lightgbm.features.store import FEATURE_BUILDER_VERSION
from src.models.results import result_path

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 120)

SOURCE_GROUP = 'Pseudo'
CATEGORY_ID = 890
EASTER_ECHO_ORIGIN = '2026-04-13'
POST_EVENT_ORIGINS = ['2026-04-06', '2026-04-13', '2026-04-20', '2026-05-04', '2026-06-01']

design = load_benchmark_design()
new_forecast_path = result_path(TWO_STAGE_MODEL_NAME, design)
old_forecast_path = ROOT / 'reports' / 'results' / 'backup_position_lift_shrinkage' / 'forecasts_two_stage.csv'

con = duckdb.connect()
con.execute('PRAGMA threads=4')
for label, path in [('old', old_forecast_path), ('new', new_forecast_path)]:
    con.execute(
        f'''
        CREATE OR REPLACE TEMP TABLE forecasts_{label} AS
        SELECT
            ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
            sourcing_group, category_id::INTEGER AS category_id,
            CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
            actual::DOUBLE AS actual, forecast::DOUBLE AS forecast,
            occurrence_probability::DOUBLE AS occurrence_probability,
            positive_quantity_forecast::DOUBLE AS positive_quantity_forecast
        FROM read_csv_auto(?)
        WHERE is_active
        ''',
        [str(path)],
    )
alignment = con.execute('''
    SELECT
        (SELECT COUNT(*) FROM forecasts_old) AS old_rows,
        (SELECT COUNT(*) FROM forecasts_new) AS new_rows,
        (SELECT COUNT(*) FROM forecasts_old o
         INNER JOIN forecasts_new n USING (ARTIKEL_ID, MARKT_ID, origin, period)
         WHERE o.actual <> n.actual) AS actual_mismatches
''').fetchdf()
display(alignment)
assert alignment.loc[0, 'old_rows'] == alignment.loc[0, 'new_rows']
assert alignment.loc[0, 'actual_mismatches'] == 0

,old_rows,new_rows,actual_mismatches
0,2502829,2502829,0


## Verification: the features measure the contamination they should

For the 13 April origin the contaminated and clean trailing means are compared per
Pseudo 890 series. If the mechanism is real, the standard mean should exceed the
non-event mean for most series, and `event_window_share_last_24` should be high —
the trailing 24 active rows at that origin reach back through the entire
Karfreitag-to-Ostermontag double window.

In [2]:
FEATURE_STORE_ROOT = ROOT / 'data' / 'processed' / 'model_features'

def generation_for(version):
    matches = []
    for manifest_path in FEATURE_STORE_ROOT.rglob('manifest.json'):
        try:
            manifest = json.loads(manifest_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if manifest.get('signature', {}).get('feature_builder_version') == version:
            matches.append((manifest_path.stat().st_mtime, manifest_path.parent))
    if not matches:
        raise FileNotFoundError(f'No feature generation for builder version {version}')
    return max(matches)[1]

new_generation = generation_for(FEATURE_BUILDER_VERSION)
print(f'feature generation: {new_generation}')

echo_features = con.execute(
    f'''
    SELECT
        COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID)) AS series,
        AVG(rolling_24_mean) AS mean_contaminated,
        AVG(rolling_24_mean_non_event) AS mean_clean,
        AVG((rolling_24_mean > rolling_24_mean_non_event)::INTEGER)
            AS share_contaminated_above_clean,
        AVG(event_window_share_last_24) AS mean_event_share,
        QUANTILE_CONT(event_window_share_last_24, 0.5) AS median_event_share
    FROM (
        SELECT ARTIKEL_ID, MARKT_ID,
               ANY_VALUE(rolling_24_mean) AS rolling_24_mean,
               ANY_VALUE(rolling_24_mean_non_event) AS rolling_24_mean_non_event,
               ANY_VALUE(event_window_share_last_24) AS event_window_share_last_24
        FROM read_parquet(?, hive_partitioning=false)
        WHERE sourcing_group = ? AND category_id = ? AND is_active
        GROUP BY 1, 2
    )
    ''',
    [str(new_generation / f'origin={EASTER_ECHO_ORIGIN}' / 'features.parquet'),
     SOURCE_GROUP, CATEGORY_ID],
).fetchdf()
display(echo_features.style.format({
    'series': '{:,.0f}', 'mean_contaminated': '{:.3f}', 'mean_clean': '{:.3f}',
    'share_contaminated_above_clean': '{:.1%}',
    'mean_event_share': '{:.1%}', 'median_event_share': '{:.1%}',
}))

feature generation: /Users/vlada/UNI/SoSe2026/ba/ba_code/data/processed/model_features/lightgbm_daily/v1/b861e82dfda7fcbed5f3


,series,mean_contaminated,mean_clean,share_contaminated_above_clean,mean_event_share,median_event_share
0,"18,559",1.314,1.230,50.8%,29.2%,29.2%


## Pseudo 890: per-origin comparison

Baseline ("old") is the `05_07` shrinkage run. The rows to watch are the
post-event weeks (2026-04-13 above all, and 2026-04-06/04-20/05-04/06-01 as
weeks whose trailing windows still contain event days).

In [3]:
def origin_metrics(label):
    return con.execute(f'''
        WITH article_day AS (
            SELECT ARTIKEL_ID, origin, period,
                   SUM(actual) AS actual, SUM(forecast) AS forecast
            FROM forecasts_{label}
            WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
            GROUP BY 1, 2, 3
        )
        SELECT origin, SUM(actual) AS actual_kg,
               SUM(ABS(forecast - actual)) AS absolute_error_kg,
               SUM(ABS(forecast - actual)) / NULLIF(SUM(actual), 0) AS wape,
               SUM(forecast) / NULLIF(SUM(actual), 0) AS ratio
        FROM article_day GROUP BY origin ORDER BY origin
    ''').fetchdf()

pseudo_890 = origin_metrics('old').merge(
    origin_metrics('new'), on='origin', suffixes=('_old', '_new'), validate='one_to_one'
)
pseudo_890 = pseudo_890.drop(columns=['actual_kg_new']).rename(columns={'actual_kg_old': 'actual_kg'})
pseudo_890['wape_change_pp'] = 100 * (pseudo_890.wape_new - pseudo_890.wape_old)
pseudo_890['post_event_week'] = pseudo_890.origin.astype(str).isin(POST_EVENT_ORIGINS)
display(pseudo_890.style.format({
    'origin': '{:%Y-%m-%d}', 'actual_kg': '{:,.0f}',
    'absolute_error_kg_old': '{:,.0f}', 'absolute_error_kg_new': '{:,.0f}',
    'wape_old': '{:.2%}', 'wape_new': '{:.2%}', 'wape_change_pp': '{:+.2f}',
    'ratio_old': '{:.3f}', 'ratio_new': '{:.3f}',
}))

def scope_wape(mask):
    return (
        pseudo_890.loc[mask, 'absolute_error_kg_old'].sum() / pseudo_890.loc[mask, 'actual_kg'].sum(),
        pseudo_890.loc[mask, 'absolute_error_kg_new'].sum() / pseudo_890.loc[mask, 'actual_kg'].sum(),
    )
scopes = pd.DataFrame(
    [
        ('all 20 origins', *scope_wape(pseudo_890.post_event_week.notna())),
        ('post-event weeks', *scope_wape(pseudo_890.post_event_week)),
        ('other weeks', *scope_wape(~pseudo_890.post_event_week)),
    ],
    columns=['scope', 'wape_old', 'wape_new'],
)
scopes['wape_change_pp'] = 100 * (scopes.wape_new - scopes.wape_old)
display(scopes.style.format({
    'wape_old': '{:.2%}', 'wape_new': '{:.2%}', 'wape_change_pp': '{:+.2f}',
}))

,origin,actual_kg,absolute_error_kg_old,wape_old,ratio_old,absolute_error_kg_new,wape_new,ratio_new,wape_change_pp,post_event_week
0,2026-03-02,"147,760","33,386",22.59%,0.951,"33,606",22.74%,0.941,+0.15,False
1,2026-03-09,"128,642","28,415",22.09%,1.097,"29,035",22.57%,1.088,+0.48,False
2,2026-03-16,"127,959","24,156",18.88%,1.041,"26,017",20.33%,1.040,+1.45,False
3,2026-03-23,"115,017","24,542",21.34%,1.089,"26,873",23.36%,1.120,+2.03,False
4,2026-03-30,"178,734","42,597",23.83%,0.872,"41,567",23.26%,0.903,-0.58,False
5,2026-04-06,"106,989","21,162",19.78%,1.095,"21,548",20.14%,1.092,+0.36,True
6,2026-04-13,"124,003","42,060",33.92%,1.289,"43,576",35.14%,1.314,+1.22,True
7,2026-04-20,"123,267","28,932",23.47%,0.925,"29,180",23.67%,0.913,+0.20,True
8,2026-04-27,"151,230","43,190",28.56%,0.797,"41,758",27.61%,0.814,-0.95,False
9,2026-05-04,"131,903","30,140",22.85%,0.881,"30,445",23.08%,0.880,+0.23,True


,scope,wape_old,wape_new,wape_change_pp
0,all 20 origins,25.22%,25.23%,+0.01
1,post-event weeks,24.88%,25.25%,+0.37
2,other weeks,25.32%,25.22%,-0.10


## Daily view of the week after Easter

In [4]:
echo_daily = con.execute(f'''
    WITH old_day AS (
        SELECT period, SUM(actual) AS actual_kg, SUM(forecast) AS forecast_old
        FROM (SELECT ARTIKEL_ID, period, SUM(actual) AS actual, SUM(forecast) AS forecast
              FROM forecasts_old
              WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
                    AND origin = DATE '{EASTER_ECHO_ORIGIN}'
              GROUP BY 1, 2)
        GROUP BY 1
    ), new_day AS (
        SELECT period, SUM(forecast) AS forecast_new
        FROM (SELECT ARTIKEL_ID, period, SUM(forecast) AS forecast
              FROM forecasts_new
              WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
                    AND origin = DATE '{EASTER_ECHO_ORIGIN}'
              GROUP BY 1, 2)
        GROUP BY 1
    )
    SELECT o.period, o.actual_kg, o.forecast_old, n.forecast_new,
           o.forecast_old / NULLIF(o.actual_kg, 0) AS ratio_old,
           n.forecast_new / NULLIF(o.actual_kg, 0) AS ratio_new
    FROM old_day AS o INNER JOIN new_day AS n USING (period)
    ORDER BY o.period
''').fetchdf()
echo_daily['period'] = pd.to_datetime(echo_daily.period)
echo_daily.insert(1, 'weekday', echo_daily.period.dt.day_name().str[:3])
display(echo_daily.style.format({
    'period': '{:%Y-%m-%d}', 'actual_kg': '{:,.0f}',
    'forecast_old': '{:,.0f}', 'forecast_new': '{:,.0f}',
    'ratio_old': '{:.3f}', 'ratio_new': '{:.3f}',
}))

,period,weekday,actual_kg,forecast_old,forecast_new,ratio_old,ratio_new
0,2026-04-13,Mon,"16,207","22,280","22,020",1.375,1.359
1,2026-04-14,Tue,"14,681","19,256","19,522",1.312,1.330
2,2026-04-15,Wed,"15,056","18,948","19,197",1.259,1.275
3,2026-04-16,Thu,"18,719","24,965","25,158",1.334,1.344
4,2026-04-17,Fri,"27,469","32,603","34,660",1.187,1.262
5,2026-04-18,Sat,"31,871","41,839","42,378",1.313,1.330


## Guardrail: portfolio segments and target-metric grains

In [5]:
def segment_metrics(label):
    return con.execute(f'''
        WITH article_day AS (
            SELECT sourcing_group || ' ' || category_id AS segment,
                   ARTIKEL_ID, origin, period,
                   SUM(actual) AS actual, SUM(forecast) AS forecast
            FROM forecasts_{label}
            GROUP BY 1, 2, 3, 4
        ), by_segment AS (
            SELECT segment, SUM(actual) AS actual_kg,
                   SUM(ABS(forecast - actual)) AS error_kg, SUM(forecast) AS forecast_kg
            FROM article_day GROUP BY 1
        )
        SELECT segment, actual_kg, error_kg, forecast_kg FROM by_segment
        UNION ALL
        SELECT 'TOTAL', SUM(actual_kg), SUM(error_kg), SUM(forecast_kg) FROM by_segment
    ''').fetchdf()

portfolio = segment_metrics('old').merge(
    segment_metrics('new'), on='segment', suffixes=('_old', '_new'), validate='one_to_one'
)
portfolio['wape_old'] = portfolio.error_kg_old / portfolio.actual_kg_old
portfolio['wape_new'] = portfolio.error_kg_new / portfolio.actual_kg_new
portfolio['wape_change_pp'] = 100 * (portfolio.wape_new - portfolio.wape_old)
portfolio['bias_old'] = portfolio.forecast_kg_old / portfolio.actual_kg_old - 1
portfolio['bias_new'] = portfolio.forecast_kg_new / portfolio.actual_kg_new - 1
display(portfolio[[
    'segment', 'actual_kg_old', 'wape_old', 'wape_new', 'wape_change_pp',
    'bias_old', 'bias_new',
]].rename(columns={'actual_kg_old': 'actual_kg'}).style.format({
    'actual_kg': '{:,.0f}', 'wape_old': '{:.2%}', 'wape_new': '{:.2%}',
    'wape_change_pp': '{:+.2f}', 'bias_old': '{:+.2%}', 'bias_new': '{:+.2%}',
}))

GRAIN_QUERIES = {
    'row (article-store-day)': 'SELECT actual, forecast FROM forecasts_{label}',
    'article-store-week': ('SELECT SUM(actual) AS actual, SUM(forecast) AS forecast '
                           'FROM forecasts_{label} GROUP BY ARTIKEL_ID, MARKT_ID, origin'),
    'article-day (stores pooled)': ('SELECT SUM(actual) AS actual, SUM(forecast) AS forecast '
                                    'FROM forecasts_{label} GROUP BY ARTIKEL_ID, origin, period'),
}
grain_rows = []
for grain, query in GRAIN_QUERIES.items():
    row = {'grain': grain}
    for label in ('old', 'new'):
        wape, bias = con.execute(
            'SELECT SUM(ABS(forecast - actual)) / SUM(actual), '
            'SUM(forecast - actual) / SUM(actual) '
            f'FROM ({query.format(label=label)})'
        ).fetchone()
        row[f'wape_{label}'] = wape
        row[f'bias_{label}'] = bias
    grain_rows.append(row)
grains = pd.DataFrame(grain_rows)
grains['wape_change_pp'] = 100 * (grains.wape_new - grains.wape_old)
display(grains[['grain', 'wape_old', 'wape_new', 'wape_change_pp', 'bias_old', 'bias_new']]
        .style.format({
    'wape_old': '{:.2%}', 'wape_new': '{:.2%}', 'wape_change_pp': '{:+.2f}',
    'bias_old': '{:+.2%}', 'bias_new': '{:+.2%}',
}))

,segment,actual_kg,wape_old,wape_new,wape_change_pp,bias_old,bias_new
0,Pseudo 890,"2,594,623",25.22%,25.23%,+0.01,-1.09%,-0.69%
1,Pseudo 900,"2,950",40.52%,41.16%,+0.64,+13.93%,+13.17%
2,FCM 890,"4,248",90.04%,90.81%,+0.78,+48.23%,+48.75%
3,FCM 900,"59,637",25.02%,24.53%,-0.49,+4.29%,+3.74%
4,TOTAL,"2,661,458",25.34%,25.34%,+0.00,-0.87%,-0.50%


,grain,wape_old,wape_new,wape_change_pp,bias_old,bias_new
0,row (article-store-day),58.28%,58.34%,+0.06,-0.87%,-0.50%
1,article-store-week,37.87%,37.94%,+0.07,-0.87%,-0.50%
2,article-day (stores pooled),25.34%,25.34%,+0.00,-0.87%,-0.50%


## Did the model use the new features?

In [6]:
new_importance = pd.read_csv(
    result_path(TWO_STAGE_MODEL_NAME, design, artifact='feature_importance')
)
recent_features = [
    'rolling_24_mean', 'rolling_24_mean_non_event', 'event_window_share_last_24',
    'rolling_6_mean',
]
recent_importance = (
    new_importance.loc[new_importance.feature.isin(recent_features)]
    .assign(rank=lambda frame: frame.groupby(['evaluation_origin', 'stage'])
            .gain_share.rank(ascending=False))
    .groupby(['stage', 'feature'])
    .agg(mean_gain_share=('gain_share', 'mean'),
         max_gain_share=('gain_share', 'max'))
    .reset_index()
    .sort_values(['stage', 'mean_gain_share'], ascending=[True, False])
)
display(recent_importance.style.format({
    'mean_gain_share': '{:.3%}', 'max_gain_share': '{:.3%}',
}))

,stage,feature,mean_gain_share,max_gain_share
3,occurrence,rolling_6_mean,0.111%,0.125%
2,occurrence,rolling_24_mean_non_event,0.041%,0.058%
1,occurrence,rolling_24_mean,0.007%,0.012%
0,occurrence,event_window_share_last_24,0.005%,0.014%
7,positive_quantity,rolling_6_mean,0.161%,0.211%
5,positive_quantity,rolling_24_mean,0.062%,0.077%
6,positive_quantity,rolling_24_mean_non_event,0.056%,0.076%
4,positive_quantity,event_window_share_last_24,0.039%,0.091%


## Conclusions

**The hypothesized mechanism exists but is too small to carry the target week,
and the batch is a wash on WAPE with a further bias gain.**

- **Verification first**: at the 13 April origin the contaminated trailing mean
  (1.314) exceeds the clean non-event mean (1.230) by only ~7% on average, and
  only 50.8% of Pseudo 890 series are inflated at all. The `05_04` §8 diagnosis
  was directionally right but quantitatively minor: the trailing-window echo can
  explain a few points of the 13 April overforecast, not the ~30% level miss.
- **The target week did not improve**: 2026-04-13 WAPE 33.92% → 35.14%
  (ratio 1.289 → 1.314). Post-event weeks as a group are +0.37 pp. Overall
  Pseudo 890 and portfolio WAPE are exactly flat (25.34% → 25.34%), with mixed
  noise-level movement across other origins (2026-06-15 −1.94 pp,
  2026-05-18 −1.69 pp versus 2026-03-23 +2.03 pp).
- **Bias improved again**: total bias −0.87% → **−0.50%** (Pseudo 890
  −1.09% → −0.69%), continuing the march toward zero across the three batches
  (−1.97% → −0.96% → −0.87% → −0.50%).
- **The features are used, modestly**: `rolling_24_mean_non_event` out-gains the
  contaminated `rolling_24_mean` in the occurrence stage and roughly matches it
  in the quantity stage; `event_window_share_last_24` receives small but nonzero
  gain. The model redistributed level information between the two means rather
  than discovering new signal — consistent with the small measured contamination.

**Revised understanding of the 13 April overforecast.** The `05_04` historical
table showed week-after-Easter mean demand of 1.552 kg/row in the 2025 fit
origins versus 1.114 in the 2026 evaluation — the post-Easter lull was simply
much deeper in 2026 than in the year the annual features (and the training
targets) describe. That is a genuine year-over-year level shift, which trailing
or annual features cannot anticipate by construction. Fixing it would need a
dedicated post-event-week lift estimated like the position lift (relative to a
non-event baseline rather than to last year's level) — a candidate for a later
batch.

**Decision: keep the batch.** It costs nothing on WAPE, moves bias meaningfully
toward zero (the stated goal), and the clean level estimate is a safer input for
any future event-related feature work.

**Scoreboard after three batches** (baseline = pre-05_06 run):
article-day WAPE 26.01% → **25.34%**, article-store-week 38.09% → **37.94%**,
row-level 58.68% → **58.34%**, total bias −1.97% → **−0.50%**.